<a href="https://colab.research.google.com/github/JaafarBK02/satellite-cnn-vit/blob/main/02_train_tile2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, numpy as np, torch, torchvision
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Drive (cache lives here) ---
from google.colab import drive
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/tile2vec_data/features.npz'

# --- Repo, tiles, checkpoint: only fetched if missing ---
if not os.path.exists('tile2vec'):
    !git clone -q https://github.com/ermongroup/tile2vec.git
if not os.path.exists('tile2vec/data/tiles/1tile.npy'):
    !cd tile2vec/data && unzip -nq tiles.zip
ckpt = 'tile2vec/models/naip_trained.ckpt'
if not os.path.exists(ckpt) or os.path.getsize(ckpt) < 1e6:        # missing, or not the real weights
    !wget -q -O tile2vec/models/naip_trained.ckpt "https://www.dropbox.com/s/bvzriiqlcof5lol/naip_trained.ckpt?dl=1"

if os.path.exists(CACHE):
    # --- Fast path: load saved features ---
    d = np.load(CACHE)
    X, y, X_raw, X_imagenet = d['X'], d['y'], d['X_raw'], d['X_imagenet']
    print('Loaded cached features from Drive')
else:
    # --- First time: build everything, then save ---
    tiles = np.stack([np.load(f'tile2vec/data/tiles/{i+1}tile.npy') for i in range(1000)])  # (1000,50,50,5)
    y = np.load('tile2vec/data/tiles/y.npy')
    X_raw = tiles[..., :4].reshape(1000, -1) / 255.0

    # Pretrained Tile2Vec (batched version of the old cell 12)
    sys.path.append('tile2vec')
    from src.tilenet import make_tilenet
    tilenet = make_tilenet(in_channels=4, z_dim=512)
    tilenet.load_state_dict(torch.load(ckpt, map_location=device), strict=False)
    tilenet.to(device).eval()
    t4 = torch.from_numpy(np.moveaxis(tiles[..., :4].astype(np.float32), -1, 1) / 255.0)
    with torch.no_grad():
        X = np.concatenate([tilenet.encode(t4[b:b+100].to(device)).cpu().numpy() for b in range(0, 1000, 100)])

    # ImageNet ResNet-18 (RGB)
    resnet = torchvision.models.resnet18(weights='IMAGENET1K_V1')
    resnet.fc = torch.nn.Identity()
    resnet = resnet.to(device).eval()
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    rgb = t4[:, :3]
    with torch.no_grad():
        X_imagenet = np.concatenate([
            resnet((F.interpolate(rgb[b:b+100].to(device), size=224, mode='bilinear', align_corners=False) - mean) / std).cpu().numpy()
            for b in range(0, 1000, 100)])

    np.savez(CACHE, X=X, y=y, X_raw=X_raw, X_imagenet=X_imagenet)
    print('Built features and saved to Drive')

# --- Evaluation helpers (definitions only, nothing slow runs) ---
def evaluate_trials(X, y, make_model, n_trials=100, n_train=None):
    accs, f1s = [], []
    for i in range(n_trials):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=i)
        if n_train is not None:
            X_tr, y_tr = X_tr[:n_train], y_tr[:n_train]
        p = make_model(i).fit(X_tr, y_tr).predict(X_te)
        accs.append(accuracy_score(y_te, p))
        f1s.append(f1_score(y_te, p, average='macro', zero_division=0))
    return np.array(accs), np.array(f1s)

rf     = lambda i: RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=i)
lr     = lambda i: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
pca_rf = lambda i: make_pipeline(PCA(10, random_state=i), rf(i))
pca_lr = lambda i: make_pipeline(PCA(10, random_state=i), StandardScaler(), LogisticRegression(max_iter=2000))
setups = {
    'Tile2Vec (pretrained)':    (X,          rf,     lr),
    'ImageNet ResNet-18 (RGB)': (X_imagenet, rf,     lr),
    'Raw pixels + PCA(10)':     (X_raw,      pca_rf, pca_lr),
}
print('Ready:', X.shape, X_raw.shape, X_imagenet.shape, y.shape)

Mounted at /content/drive
Loaded cached features from Drive
Ready: (1000, 512) (1000, 10000) (1000, 512) (1000,)


In [2]:
print(np.allclose(X, np.load('/content/drive/MyDrive/tile2vec_data/X_pretrained.npy'), atol=1e-4))

True


In [3]:
for n in [50, 100]:
    for clf, pick in [('RF', 1), ('LR', 2)]:
        a_t2v, f_t2v = evaluate_trials(X,          y, setups['Tile2Vec (pretrained)'][pick],    n_trials=100, n_train=n)
        a_imn, f_imn = evaluate_trials(X_imagenet, y, setups['ImageNet ResNet-18 (RGB)'][pick], n_trials=100, n_train=n)
        print(f'n={n:3d} {clf}: Tile2Vec {a_t2v.mean():.4f} vs ImageNet {a_imn.mean():.4f} | '
              f'Tile2Vec better on {np.mean(a_t2v > a_imn):.0%} | gain {np.mean(a_t2v - a_imn):+.4f} | '
              f'Wilcoxon p = {wilcoxon(a_t2v, a_imn).pvalue:.2e} | macro F1 {f_t2v.mean():.3f} vs {f_imn.mean():.3f}')

n= 50 RF: Tile2Vec 0.5402 vs ImageNet 0.5810 | Tile2Vec better on 17% | gain -0.0408 | Wilcoxon p = 5.55e-12 | macro F1 0.154 vs 0.145
n= 50 LR: Tile2Vec 0.5094 vs ImageNet 0.5905 | Tile2Vec better on 8% | gain -0.0810 | Wilcoxon p = 4.26e-17 | macro F1 0.162 vs 0.170
n=100 RF: Tile2Vec 0.5964 vs ImageNet 0.6144 | Tile2Vec better on 31% | gain -0.0181 | Wilcoxon p = 2.24e-05 | macro F1 0.191 vs 0.167
n=100 LR: Tile2Vec 0.5644 vs ImageNet 0.6170 | Tile2Vec better on 12% | gain -0.0526 | Wilcoxon p = 2.65e-15 | macro F1 0.200 vs 0.196


In [4]:
!ls -lh /content/drive/MyDrive/tile2vec_data
!ls tile2vec/src
!grep -n "def \|class " tile2vec/src/*.py
!grep -n "margin\|l2\|lr\|epochs\|batch_size\|z_dim\|train_triplet_epoch\|triplet_dataloader\|TileNet_epoch" "tile2vec/examples/Example 2 - Train Tile2Vec from scratch.ipynb"

total 2.3G
drwx------ 2 root root 4.0K Nov 19 19:02 checkpoints
-rw------- 1 root root  81M Nov 19 04:29 features.npz
-rw------- 1 root root 2.2G Nov 18 16:59 triplets.zip
-rw------- 1 root root 4.0M Nov 19 04:21 X_pretrained.npy
cdl.py	     data_utils.py  sample_tiles.py  training.py
datasets.py  resnet.py	    tilenet.py
tile2vec/src/datasets.py:10:class TileTripletsDataset(Dataset):
tile2vec/src/datasets.py:12:    def __init__(self, tile_dir, transform=None, n_triplets=None,
tile2vec/src/datasets.py:20:    def __len__(self):
tile2vec/src/datasets.py:24:    def __getitem__(self, idx):
tile2vec/src/datasets.py:44:class GetBands(object):
tile2vec/src/datasets.py:48:    def __init__(self, bands):
tile2vec/src/datasets.py:52:    def __call__(self, sample):
tile2vec/src/datasets.py:59:class RandomFlipAndRotate(object):
tile2vec/src/datasets.py:65:    def __call__(self, sample):
tile2vec/src/datasets.py:85:class ClipAndScale(object):
tile2vec/src/datasets.py:90:    def __init__(self, img_ty

In [5]:
import os
def find_triplet_dir():
    dirs = [(r, len(f)) for r, _, f in os.walk('/content')
            if '0anchor.npy' in f and '/drive' not in r and 'example_tiles' not in r]
    return max(dirs, key=lambda d: d[1])[0] + '/' if dirs else None

TRIPLET_DIR = find_triplet_dir()
if TRIPLET_DIR is None:
    print('Unzipping triplets.zip to local disk (a few minutes)...')
    !unzip -nq /content/drive/MyDrive/tile2vec_data/triplets.zip -d /content/
    TRIPLET_DIR = find_triplet_dir()

assert TRIPLET_DIR is not None, 'No triplets found after unzip: check that triplets.zip is complete'
n_files = len(os.listdir(TRIPLET_DIR))
print(TRIPLET_DIR, n_files, 'files ->', n_files // 3, 'triplets')

Unzipping triplets.zip to local disk (a few minutes)...
/content/triplets/ 300000 files -> 100000 triplets


In [ ]:
import os, sys, glob, time, inspect, types, importlib, torch
from torch import optim

# --- Compatibility patches for the authors' 2018 code (safe to rerun) ---
!sed -i 's/\.data\[0\]/.item()/g' tile2vec/src/training.py      # PyTorch 0.3 -> modern: loss.data[0] -> loss.item()
sys.path.append('tile2vec')
try:
    from osgeo import gdal                                         # 'import gdal' no longer exists in modern GDAL
    sys.modules['gdal'] = gdal
except ImportError:
    sys.modules['gdal'] = types.ModuleType('gdal')                 # placeholder: training never uses gdal

from src.tilenet import make_tilenet
from src.datasets import triplet_dataloader
import src.training
importlib.reload(src.training)                                     # make sure the patched file is used
from src.training import train_triplet_epoch

cuda = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if cuda else 'NONE - switch runtime to GPU!')
torch.manual_seed(0)
CKPT_DIR = '/content/drive/MyDrive/tile2vec_data/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# --- Authors' hyperparameters (Example 2) ---
bands, z_dim, batch_size, lr = 4, 512, 50, 1e-3
margin, l2, EPOCHS = 10, 0.01, 50
KEEP = {1, 5, 10, 20, 30, 40, 50}        # checkpoints kept for the epochs-vs-accuracy analysis

# Only pass optional arguments the authors' functions actually accept
dl_extra = {k: v for k, v in dict(n_triplets=100000, pairs_only=True).items()
            if k in inspect.signature(triplet_dataloader).parameters}
dataloader = triplet_dataloader('naip', TRIPLET_DIR, bands=bands, augment=True, batch_size=batch_size,
                                shuffle=True, num_workers=2, **dl_extra)   # Colab has 2 CPUs

model = make_tilenet(in_channels=bands, z_dim=z_dim)
if cuda: model.cuda()
optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.5, 0.999))

# --- Resume from the latest checkpoint, if any ---
epoch_of = lambda p: int(p.split('epoch')[1].split('.')[0])
ckpts = sorted(glob.glob(f'{CKPT_DIR}/ours_epoch*.pt'), key=epoch_of)
start, history = 0, []
if ckpts:
    s = torch.load(ckpts[-1], map_location='cuda' if cuda else 'cpu', weights_only=False)
    model.load_state_dict(s['model']); optimizer.load_state_dict(s['optimizer'])
    start, history = s['epoch'], s['history']
    print(f'Resumed after epoch {start}')

tr_extra = {k: v for k, v in dict(print_every=500, t0=time.time()).items()
            if k in inspect.signature(train_triplet_epoch).parameters}

# --- Training loop: one checkpoint to Drive per epoch ---
model.train()
for epoch in range(start, EPOCHS):
    te = time.time()
    avg_loss, avg_l_n, avg_l_d, avg_l_nd = train_triplet_epoch(
        model, cuda, dataloader, optimizer, epoch + 1, margin=margin, l2=l2, **tr_extra)
    history.append(dict(epoch=epoch + 1, loss=avg_loss, l_n=avg_l_n, l_d=avg_l_d, l_nd=avg_l_nd,
                        minutes=(time.time() - te) / 60))
    path = f'{CKPT_DIR}/ours_epoch{epoch + 1}.pt'
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'epoch': epoch + 1, 'history': history}, path)
    prev = f'{CKPT_DIR}/ours_epoch{epoch}.pt'                      # keep milestones only, to save Drive space
    if epoch not in KEEP and os.path.exists(prev):
        os.remove(prev)
    print(f'Epoch {epoch + 1}/{EPOCHS}: loss {avg_loss:.4f} | {history[-1]["minutes"]:.1f} min')

GPU: Tesla T4
Epoch 1: [500/100000 (0%)], Avg loss: 9.2344
Epoch 1: [1000/100000 (1%)], Avg loss: 7.6899
Epoch 1: [1500/100000 (2%)], Avg loss: 6.8314
Epoch 1: [2000/100000 (2%)], Avg loss: 6.6965
Epoch 1: [2500/100000 (2%)], Avg loss: 6.7128
Epoch 1: [3000/100000 (3%)], Avg loss: 6.2017
Epoch 1: [3500/100000 (4%)], Avg loss: 6.3546
Epoch 1: [4000/100000 (4%)], Avg loss: 6.6722
Epoch 1: [4500/100000 (4%)], Avg loss: 6.2268
Epoch 1: [5000/100000 (5%)], Avg loss: 5.7718
Epoch 1: [5500/100000 (6%)], Avg loss: 5.9551
Epoch 1: [6000/100000 (6%)], Avg loss: 6.3049
Epoch 1: [6500/100000 (6%)], Avg loss: 5.6641
Epoch 1: [7000/100000 (7%)], Avg loss: 5.8203
Epoch 1: [7500/100000 (8%)], Avg loss: 5.3755
Epoch 1: [8000/100000 (8%)], Avg loss: 5.4250
Epoch 1: [8500/100000 (8%)], Avg loss: 6.0914
Epoch 1: [9000/100000 (9%)], Avg loss: 5.9808
Epoch 1: [9500/100000 (10%)], Avg loss: 5.3826
Epoch 1: [10000/100000 (10%)], Avg loss: 5.6047
Epoch 1: [10500/100000 (10%)], Avg loss: 5.8835
Epoch 1: [11000/